# 006-segmentation-ceiling

# Imports

In [ ]:
import json
import os

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from gt_query import QueryCorrelator, gt_labels, gt_query, damaged, substituted
from uavloc.data.drone_frames import DroneFrame, load_frames
from uavloc.data.multidate import MULTIDATE, load_date
from uavloc.datasets import dataset_ref
from uavloc.features.factory import correlator_config
from uavloc.features.segmentation import CLASSES, SegmentationFeatures, label_probabilities, load_segmenter
from uavloc.geo import footprint_m
from uavloc.matching.benchmark import run_benchmark, summarise
from uavloc.matching.correlator import CAMERA_ASPECT_W_OVER_H, drone_class_shares
from uavloc.track import REPO_ROOT, experiment_dir, log_run, set_seed

# Params

In [ ]:
EXP = "006-segmentation-ceiling"
QUERY_SOURCE = "model"            # "model" | "gt": where the "drone" class map comes from
MAP_SOURCE = "model"              # "model" | "gt": where the satellite class map comes from
GT_CLASSES = None                 # None | "all" (with QUERY_SOURCE "gt") | class names whose query channel is taken from GT
DAMAGE = "none"                   # "none" | "stride" | "boundary", applied to the GT query
DAMAGE_PX = 0                     # stride s, or boundary radius r in px
LABEL_SOURCE = "osm"              # "osm" (pre-registered) | "annotated" (hand-corrected, current date)
NOTE = ""

PLACES = ["bushcha-50p2422-25p8305", "birky-49p9105-23p9192"]
MAP_DATE = "newest"
MATCHING = "weighted-ncc"
CLASS_WEIGHTS = (0.0, 1.0, 1.0, 0.5, 0.0, 0.5)   # background, building, track, woody, field, water
SEARCH_SCALE = 1.7
NOISE_M = [5.0, 10.0, 15.0]
PRIMARY_NOISE_M = 15.0
N_DIRECTIONS = 8
SEED = 42
DEVICE = "mps"
FOV_W_DEG = 60.0
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0
SEG_ARCHITECTURE = "lraspp"
SEG_CHECKPOINT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHANNELS = "all"
MATCH_GSD_M = 0.385
FRAMES_DATASET = "village-crossdate-frames"
LABEL_DATASETS = {"osm": MULTIDATE, "annotated": "ua-village-annotated"}

# Data

In [ ]:
set_seed(SEED)

In [ ]:
if QUERY_SOURCE == "model" and DAMAGE != "none":
    raise ValueError("damage applies to the GT query only")
GT_CLASS_IDS = [CLASSES.index(c) for c in GT_CLASSES] if isinstance(GT_CLASSES, list) else []

In [ ]:
RUN_CONFIG = {
    "method": "segmentation", "matching": MATCHING, "class_weights": CLASS_WEIGHTS,
    "noise_m": NOISE_M, "primary_noise_m": PRIMARY_NOISE_M, "n_directions": N_DIRECTIONS,
    "search_scale": SEARCH_SCALE, "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
    "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_sat": SEG_CHECKPOINT, "seg_checkpoint_drone": SEG_CHECKPOINT,
    "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": MATCH_GSD_M, "match_gsd_m": MATCH_GSD_M,
    "frames_dataset": FRAMES_DATASET, "frames_dataset_ref": dataset_ref(FRAMES_DATASET),
    "places": PLACES, "map_date": MAP_DATE, "min_peak_z": None,
    "query_source": QUERY_SOURCE, "map_source": MAP_SOURCE, "gt_classes": GT_CLASSES,
    "damage": DAMAGE, "damage_px": DAMAGE_PX,
    "label_source": LABEL_SOURCE, "label_dataset_ref": dataset_ref(LABEL_DATASETS[LABEL_SOURCE]),
}

In [ ]:
def frame_place(frame: DroneFrame) -> str:
    return json.loads((frame.image_path.parent / "meta.json").read_text())["place"]

In [ ]:
frames = [f for f in load_frames(FRAMES_DATASET) if frame_place(f) in PLACES]
places = {place: [f for f in frames if frame_place(f) == place] for place in PLACES}
maps = {place: load_date(place, MAP_DATE) for place in PLACES}
labels = {place: gt_labels(place, LABEL_SOURCE) for place in PLACES}
{place: len(members) for place, members in places.items()}

# Model

In [ ]:
model = load_segmenter(REPO_ROOT / SEG_CHECKPOINT, DEVICE, SEG_ARCHITECTURE)

In [ ]:
def map_features(place: str) -> SegmentationFeatures:
    """The satellite class map: the model on the map date, or GT one-hot area-averaged to the match grid."""
    if MAP_SOURCE == "model":
        return SegmentationFeatures(model, model, maps[place], SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)
    return SegmentationFeatures.from_satellite_probabilities(model, label_probabilities(labels[place], MATCH_GSD_M),
                                                             SEG_CHANNELS, MATCH_GSD_M, MATCH_GSD_M, DEVICE)

In [ ]:
features = {place: map_features(place) for place in PLACES}

In [ ]:
def query_map(frame: DroneFrame, place: str) -> np.ndarray:
    """The "drone" class map `(C, H, W)`: GT cut at the true pose and damaged, or the model's with GT_CLASS_IDS from GT."""
    footprint = footprint_m(frame.altitude_m, FOV_W_DEG, CAMERA_ASPECT_W_OVER_H)
    gt = gt_query(labels[place], frame, footprint, MATCH_GSD_M)
    if QUERY_SOURCE == "gt":
        return damaged(gt, DAMAGE, DAMAGE_PX)
    return substituted(features[place].drone_probabilities(frame.image(), *footprint), gt, GT_CLASS_IDS)

In [ ]:
correlators = {place: QueryCorrelator(features[place], correlator_config(RUN_CONFIG),
                                      lambda frame, place=place: query_map(frame, place))
               for place in PLACES}

# Run

In [ ]:
records = pd.concat([run_benchmark(correlators[place], places[place], noise_m, N_DIRECTIONS, SEED).assign(place=place)
                     for place in PLACES for noise_m in NOISE_M], ignore_index=True)
records["kept_prior"] = records["peak_z"] == 0
records.groupby(["place", "noise_m"])[["error_m", "kept_prior"]].mean().round(2)

In [ ]:
query_shares = pd.DataFrame({f.name: drone_class_shares(correlators[frame_place(f)].drone_map(f)) for f in frames},
                            index=CLASSES).T
query_shares.mean().round(3)

In [ ]:
def run_metrics(records: pd.DataFrame) -> dict[str, float]:
    """005's metrics, plus the share of draws that kept the prior and the query's mean class shares."""
    subsets = {"all": {f.name for f in frames}, **{p.split("-")[0]: {f.name for f in m} for p, m in places.items()}}
    metrics = summarise(records, subsets, PRIMARY_NOISE_M)
    for noise_m, group in records.groupby("noise_m"):
        metrics[f"kept_prior_share_at_{noise_m:g}m"] = float(group["kept_prior"].mean())
    metrics["kept_prior_share"] = metrics[f"kept_prior_share_at_{PRIMARY_NOISE_M:g}m"]
    metrics.update({f"query_share_{c}": float(v) for c, v in query_shares.mean().items()})
    return metrics

In [ ]:
metrics = run_metrics(records)
{k: round(v, 3) for k, v in metrics.items() if "_at_" not in k}

# Plots

In [ ]:
fig_cdf, ax = plt.subplots(figsize=(6, 4))
for noise_m, group in records.groupby("noise_m"):
    errors = np.sort(group["error_m"].to_numpy())
    line, = ax.plot(errors, np.linspace(0, 1, len(errors)), label=f"{noise_m:g} m injected")
    ax.axvline(noise_m, color=line.get_color(), ls=":", lw=1)
ax.set_xlabel("position error after correction, m"); ax.set_ylabel("share of draws")
ax.set_title(f"query {QUERY_SOURCE}, map {MAP_SOURCE}, damage {DAMAGE} {DAMAGE_PX}"); ax.legend(); ax.grid(alpha=.3)
fig_cdf.tight_layout()

In [ ]:
per_frame = records[records["noise_m"] == PRIMARY_NOISE_M].groupby("frame")["error_m"].mean().sort_values()
fig_frames, ax = plt.subplots(figsize=(12, 3.5))
per_frame.plot.bar(ax=ax, width=.8)
ax.axhline(PRIMARY_NOISE_M, color="k", ls=":", lw=1)
ax.set_ylabel(f"mean error @ {PRIMARY_NOISE_M:g} m, m"); ax.set_title("per frame, 8 directions each")
fig_frames.tight_layout()

# Log

In [ ]:
run = log_run(exp=EXP, config=RUN_CONFIG, metrics=metrics, note=NOTE, figures=[fig_cdf, fig_frames], seed=SEED, device=DEVICE)

In [ ]:
artifacts = experiment_dir(EXP) / "artifacts"
artifacts.mkdir(exist_ok=True)
records.to_csv(artifacts / f"records-{run['run']}.csv", index=False)